# NOSE LoRA + Head: molecule retrieval

Three PubChem structures are evaluated through:

`SMILES → Uni-Mol → NOSE molecular descriptor branch → cosine ranking over 1,086 text descriptors`

The trained **NOSE LoRA + Head** (512-D) text encoder ranks all descriptors against each molecular embedding. Only **odorless, neutral, slight, weak** are displayed. Rank percentile is lower-is-better.


In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress")
import logging
logging.getLogger("Uni-Mol Tools").setLevel(logging.ERROR)
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from pathlib import Path

from IPython.display import HTML, display
import numpy as np
import pandas as pd

import nose as nose_package
from nose import NOSEPipeline

ROOT = Path(nose_package.__file__).resolve().parents[1]
descriptors = (ROOT / "demos/assets/odor_descriptors.txt").read_text().splitlines()
molecules = pd.read_csv(ROOT / "demos/assets/demo_molecules.csv")
focus_terms = ["odorless", "neutral", "slight", "weak"]
assert len(descriptors) == 1086

In [2]:
import contextlib
import io

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    pipeline = NOSEPipeline.from_pretrained()
    logging.getLogger("Uni-Mol Tools").setLevel(logging.ERROR)
    nose_descriptor_embeddings = pipeline.encode_descriptors(
        descriptors,
        batch_size=32,
    )
    molecule_embeddings = pipeline.encode_smiles(
        molecules["isomeric_smiles"].tolist(),
        branch="descriptor",
    )

print(f"NOSE descriptor embeddings: {nose_descriptor_embeddings.shape}")
print(f"Molecular embeddings: {molecule_embeddings.shape}")


NOSE descriptor embeddings: torch.Size([1086, 512])
Molecular embeddings: torch.Size([3, 512])


In [3]:
scores = molecule_embeddings.numpy() @ nose_descriptor_embeddings.numpy().T
rows = []
for molecule_index, name in enumerate(molecules["name"]):
    order = np.argsort(-scores[molecule_index])
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, len(order) + 1)
    for term in focus_terms:
        descriptor_index = descriptors.index(term)
        rank = int(ranks[descriptor_index])
        rows.append({
            "molecule": name,
            "descriptor": term,
            "rank": rank,
            "rank_percentile": rank / len(descriptors) * 100,
            "cosine": float(scores[molecule_index, descriptor_index]),
        })
focus = pd.DataFrame(rows)

for name in molecules["name"]:
    subset = focus[focus["molecule"] == name].sort_values("rank")
    sequence = " <span style='color:#9ca3af;font-size:18px'>→</span> ".join(
        f"<b>{row.descriptor}</b> <span style='color:#2563eb'>{row.rank_percentile:.2f}%</span> "
        f"<small style='color:#6b7280'>(#{int(row.rank)})</small>"
        for row in subset.itertuples()
    )
    display(HTML(
        "<div style='border:1px solid #d1d5db;border-radius:10px;padding:14px 18px;"
        "margin:12px 0;background:#fafafa'>"
        f"<div style='font-size:18px;font-weight:700;margin-bottom:8px'>{name}</div>"
        f"<div style='margin:7px 0'>{sequence}</div>"
        "</div>"
    ))
